# Validación local · GAIA Lab

Este notebook mide cualquier modelo con la **métrica oficial** de Kaggle en dos validaciones temporales, **V1** y **V2**, sin fuga de datos. También calcula las líneas base que hay que superar y el error nocturno de temperatura que necesita la app.

**Cómo usarlo**
1. En Kaggle, crea el notebook dentro de la competencia, para que la data se agregue sola. Activa *GPU T4*; solo hace falta para la sección 6.
2. Corre todo de arriba abajo.
3. Para medir un modelo nuevo, sigue la sección 9.

**Reglas que respeta**
- Solo usa la data oficial de entrenamiento; no toca el test.
- Todo lo que se aprende de la data (climatología, normalización) se calcula **solo con la parte de entrenamiento** de cada validación.
- Semilla fija en 42.

## 1. Configuración y data

In [ ]:
import glob, os, time, random
from datetime import date
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def fijar_semilla(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla()
QUICK = os.environ.get('QUICK') == '1'   # solo para probar que todo corre
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
st = np.load(find('static.npz'))
Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']
print('X:', Xtr.shape, '| Y:', Ytr.shape, '| horas:', ttr[0], '→', ttr[-1], '| dispositivo:', dev)

In [ ]:
# Controles de la data
assert Xtr.shape[1:] == (17, 13, 4) and Ytr.shape[1:] == (70, 35, 4) and len(Xtr) == len(Ytr) == len(ttr)
assert np.all(ttr[:-1] < ttr[1:]), 'Las horas no están ordenadas o hay duplicados'
fechas = pd.to_datetime(ttr, format='%Y%m%d%H')   # en UTC
hora_utc = fechas.hour.values
print(f'{len(ttr)} horas; {np.isin(hora_utc, np.arange(0, 24, 3)).sum()} son de las que evalúa Kaggle (cada 3 h)')

## 2. Métrica oficial

Para cada variable, Kaggle calcula el RMSE sobre todos los píxeles de las horas 00, 03, …, 21 UTC y lo divide entre una constante: el RMSE de la climatología en el test.

```latex
score = 1 - \frac{1}{4}\left(\frac{RMSE_{RH}}{25.360} + \frac{RMSE_{T}}{2.0652} + \frac{RMSE_{U}}{2.6776} + \frac{RMSE_{V}}{2.0876}\right)
```

Mayor es mejor. Con las constantes fijas, el score local se puede comparar con el de Kaggle. El starter, en cambio, divide entre la climatología de su propia validación, y por eso su score local (0.34) no se parece al de Kaggle (0.43).

In [ ]:
CONSTANTES = np.array([25.360, 2.0652, 2.6776, 2.0876])   # RMSE de la climatología en el test: RH, T, U, V
HORAS_EVAL = np.arange(0, 24, 3)                          # 00, 03, ..., 21 UTC

def score_oficial(pred, true, horas):
    """pred y true: (n, 70, 35, 4) en unidades originales; horas: hora UTC de cada fila.
    Usa solo las horas que evalúa Kaggle. Devuelve el score y el RMSE de cada variable."""
    m = np.isin(horas, HORAS_EVAL)
    dif = pred[m].astype(np.float64) - true[m]
    rmse = np.sqrt((dif ** 2).mean(axis=(0, 1, 2)))
    return 1 - (rmse / CONSTANTES).mean(), rmse

# Control: una predicción perfecta da 1
assert abs(score_oficial(Ytr[:24], Ytr[:24], hora_utc[:24])[0] - 1) < 1e-12

## 3. Dos validaciones temporales

- **V1 (principal):** entrena hasta el 31 de mayo y valida en todo junio. El 31 de mayo no tiene datos, así que entre el entrenamiento y la validación queda un día de separación.
- **V2:** la del starter: valida del 16 al 30 de junio y entrena con todo lo anterior.

Nunca se usa un corte aleatorio: horas seguidas se parecen demasiado y darían una validación optimista.

In [ ]:
CORTES = {
    'V1': (ttr < '2024060100', ttr >= '2024060100'),
    'V2': (ttr < '2024061600', ttr >= '2024061600'),
}
for nombre, (m_tr, m_va) in CORTES.items():
    assert not (m_tr & m_va).any()
    sep = (fechas[m_va][0] - fechas[m_tr][-1]).total_seconds() / 3600
    print(f'{nombre}: entrena {m_tr.sum()} h ({ttr[m_tr][0]} → {ttr[m_tr][-1]}) · '
          f'valida {m_va.sum()} h ({ttr[m_va][0]} → {ttr[m_va][-1]}) · separación {sep:.0f} h')

## 4. Herramientas

- **Climatología:** promedio de cada hora UTC. Siempre se le pasan solo filas de entrenamiento.
- **GFS a 3 km:** interpola el GFS a la grilla del WRF con la misma grilla del starter (`grid_sample` con las coordenadas de `static.npz`). Así se respeta que en el GFS la fila 0 es el norte y en el WRF, el sur.
- **Error nocturno de T:** mide el error de temperatura en la noche de Lima (18:00–07:00, es decir, 23–12 UTC). Lo necesita la app para el margen de helada. «Puna» son las celdas de 3800 a 4800 m, el rango que usa hoy la app.

In [ ]:
def climatologia(Y, horas):
    """Promedio por hora UTC: (24, 70, 35, 4)."""
    return np.stack([Y[horas == k].mean(0) for k in range(24)])

glat, glon = st['gfs_lats'], st['gfs_lons']   # GFS: fila 0 = norte
wlat, wlon = st['wrf_lats'], st['wrf_lons']   # WRF: fila 0 = sur
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=dev)   # (1, 70, 35, 2)

def gfs_a_3km(X, bs=256):
    """Interpola el GFS (n, 17, 13, 4) a la grilla WRF (n, 70, 35, 4), en bilineal."""
    out = np.empty((len(X), 70, 35, 4), np.float32)
    with torch.no_grad():
        for s in range(0, len(X), bs):
            x = torch.tensor(X[s:s + bs], dtype=torch.float32, device=dev).permute(0, 3, 1, 2)
            y = F.grid_sample(x, GRID.expand(len(x), -1, -1, -1), align_corners=True)
            out[s:s + bs] = y.permute(0, 2, 3, 1).cpu().numpy()
    return out

PUNA = (dem >= 3800) & (dem <= 4800)

def error_nocturno_T(pred, true, fechas_utc):
    """Error absoluto de T (en °C, igual que en K) en la noche de Lima, 18:00–07:00.
    Da el error de cada hora de la noche y el de la mínima de cada noche completa (14 horas)."""
    lima = fechas_utc - pd.Timedelta(hours=5)
    hl = np.asarray(lima.hour)
    noche = (hl >= 18) | (hl <= 7)
    id_noche = np.asarray((lima - pd.to_timedelta((hl <= 7).astype(int), unit='D')).normalize())
    err = np.abs(pred[..., 1].astype(np.float64) - true[..., 1])
    res = {'horario_todas': float(err[noche].mean()), 'horario_puna': float(err[noche][:, PUNA].mean())}
    min_p, min_t = [], []
    for d in np.unique(id_noche[noche]):
        k = noche & (id_noche == d)
        if k.sum() == 14:
            min_p.append(pred[k, ..., 1].min(0)); min_t.append(true[k, ..., 1].min(0))
    res['noches_completas'] = len(min_p)
    if min_p:
        e = np.abs(np.array(min_p, np.float64) - np.array(min_t))
        res['minima_todas'], res['minima_puna'] = float(e.mean()), float(e[:, PUNA].mean())
    else:
        res['minima_todas'] = res['minima_puna'] = float('nan')
    return res

Gtr = gfs_a_3km(Xtr)
print('GFS interpolado:', Gtr.shape, '| celdas de puna (3800–4800 m):', int(PUNA.sum()))

In [ ]:
# Control de orientación: el GFS interpolado debe parecerse más al WRF que su versión volteada de norte a sur
r_ok = np.sqrt(((Gtr[..., 1] - Ytr[..., 1]) ** 2).mean())
r_volteado = np.sqrt(((Gtr[:, ::-1, :, 1] - Ytr[..., 1]) ** 2).mean())
print(f'RMSE de T: {r_ok:.2f} K bien orientado · {r_volteado:.2f} K volteado →', 'OK' if r_ok < r_volteado else 'REVISAR LA ORIENTACIÓN')

## 5. Líneas base sin entrenar

- **Climatología del corte:** el promedio de cada hora en el entrenamiento del corte. En el test da 0 por definición; en junio da más, porque junio se parece más al entrenamiento. Por eso cada experimento se compara con las líneas base del mismo corte, no con cero.
- **GFS interpolado:** el GFS llevado a 3 km, sin corregir. Puede dar un score negativo, porque el GFS no ve el relieve y falla mucho en temperatura.
- **GFS + corrección media:** el GFS interpolado más la corrección promedio de cada hora y celda (Y − GFS), aprendida en el entrenamiento. Es el enfoque nuevo en su versión más simple, sin red.

Primero, un control: en V2 y con todas las horas, la climatología debe dar el mismo RMSE que imprime el starter. Si no coincide, hay que revisar antes de seguir.

In [ ]:
m_tr, m_va = CORTES['V2']
clim = climatologia(Ytr[m_tr], hora_utc[m_tr])
rmse_v2 = np.sqrt(((clim[hora_utc[m_va]] - Ytr[m_va]) ** 2).mean(axis=(0, 1, 2)))
print('Climatología en V2, todas las horas:', {v: round(float(r), 3) for v, r in zip(VARS, rmse_v2)})
print('Debe coincidir con el starter:     ', {'RH': 17.177, 'T': 2.01, 'U': 2.261, 'V': 1.946})

In [ ]:
RESULTADOS = []

def evaluar(nombre, preds, notas=''):
    """preds: {'V1': predicción de la validación de V1, 'V2': ...}; cada una (n_val, 70, 35, 4)."""
    fila = {'Fecha': date.today().isoformat(), 'Experimento': nombre}
    for corte in ['V1', 'V2']:
        if corte in preds:
            m_va = CORTES[corte][1]
            assert preds[corte].shape == Ytr[m_va].shape, f'{corte}: forma {preds[corte].shape}, se esperaba {Ytr[m_va].shape}'
            sc, rmse = score_oficial(preds[corte], Ytr[m_va], hora_utc[m_va])
            fila[corte] = round(float(sc), 4)
            fila[f'RMSE {corte} (RH / T / U / V)'] = ' / '.join(f'{r:.3f}' for r in rmse)
    if 'V1' in preds:
        m_va = CORTES['V1'][1]
        en = error_nocturno_T(preds['V1'], Ytr[m_va], fechas[m_va])
        fila['Error mínima nocturna T en puna, V1 (°C)'] = round(en['minima_puna'], 2)
    fila['Notas'] = notas
    RESULTADOS.append(fila)
    return fila

pred_clim, pred_gfs, pred_corr = {}, {}, {}
for corte, (m_tr, m_va) in CORTES.items():
    pred_clim[corte] = climatologia(Ytr[m_tr], hora_utc[m_tr])[hora_utc[m_va]]
    pred_gfs[corte] = Gtr[m_va]
    corr = climatologia(Ytr[m_tr] - Gtr[m_tr], hora_utc[m_tr])   # corrección media por hora y celda
    pred_corr[corte] = Gtr[m_va] + corr[hora_utc[m_va]]

evaluar('Climatología del corte', pred_clim, 'línea base sin entrenar')
evaluar('GFS interpolado', pred_gfs, 'sin corrección')
evaluar('GFS + corrección media', pred_corr, 'enfoque nuevo sin red')
pd.DataFrame(RESULTADOS)

## 6. Línea base entrenada: el starter

Es el mismo modelo y el mismo entrenamiento del starter (25 épocas, AdamW y OneCycle), con tres cambios:
- semilla 42;
- estadísticas calculadas solo con el entrenamiento de cada corte;
- se evalúa el modelo de la última época, sin elegir la mejor época con la validación, para no inflar el score.

Con GPU es mucho más rápido que en CPU. En GPU puede variar en el tercer decimal entre corridas.

In [ ]:
EPOCHS = 2 if QUICK else 25
ENTRENAR_STARTER = True

def ventanas(X, t):
    """Horas t-2, t-1 y t del GFS; si falta una, repite la hora t (igual que el starter)."""
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag))
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (n, 3, 17, 13, 4)

Wtr = ventanas(Xtr, ttr)
DEM_T = torch.tensor((dem - dem.mean()) / dem.std(), dtype=torch.float32, device=dev)[None, None]
def to_t(a): return torch.tensor(a, dtype=torch.float32, device=dev)

class Net(nn.Module):   # la misma red del starter
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(
            nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, 4, 1))
    def forward(self, x, h):
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = self.enc(torch.cat([x, hh], 1))
        f = F.grid_sample(f, GRID.expand(B, -1, -1, -1), align_corners=True)
        f = torch.cat([f, DEM_T.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)

def entrenar_starter(m_tr, epochs=EPOCHS):
    fijar_semilla()
    mu_x, sd_x = Xtr[m_tr].mean((0, 1, 2)), Xtr[m_tr].std((0, 1, 2))
    clim = climatologia(Ytr[m_tr], hora_utc[m_tr])
    asd = (Ytr[m_tr] - clim[hora_utc[m_tr]]).std((0, 1, 2))
    CLIM, ASD = to_t(clim), to_t(asd)
    model = Net().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    itr = np.where(m_tr)[0]
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=epochs * int(np.ceil(len(itr) / 32)))
    for ep in range(epochs):
        model.train(); np.random.shuffle(itr); t0 = time.time()
        for s in range(0, len(itr), 32):
            b = itr[s:s + 32]
            x = to_t((Wtr[b] - mu_x) / sd_x)
            h = torch.tensor(hora_utc[b], dtype=torch.long, device=dev)
            y = (to_t(Ytr[b]) - CLIM[h]) / ASD
            loss = F.mse_loss(model(x, h.float()), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        print(f'  época {ep + 1:02d}/{epochs} · loss {loss.item():.3f} · {time.time() - t0:.0f} s')
    return model, (mu_x, sd_x, CLIM, ASD)

def predecir_starter(model, stats, idx, bs=128):
    mu_x, sd_x, CLIM, ASD = stats
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(idx), bs):
            b = idx[s:s + bs]
            x = to_t((Wtr[b] - mu_x) / sd_x)
            h = torch.tensor(hora_utc[b], dtype=torch.long, device=dev)
            out.append((CLIM[h] + model(x, h.float()) * ASD).cpu().numpy())
    return np.concatenate(out)

if ENTRENAR_STARTER:
    pred_starter = {}
    for corte, (m_tr, m_va) in CORTES.items():
        print(f'{corte}: entrenando con {m_tr.sum()} horas')
        modelo, stats = entrenar_starter(m_tr)
        pred_starter[corte] = predecir_starter(modelo, stats, np.where(m_va)[0])
    evaluar('Starter (semilla 42)', pred_starter, f'{EPOCHS} épocas, modelo de la última época')
pd.DataFrame(RESULTADOS)

## 7. Error nocturno de T para la app

El doc de umbrales usa como margen ámbar de helada el mayor valor entre 2 °C y el error nocturno de T del modelo. Aquí se mide en V1 de dos formas: el error de cada hora de la noche y el error de la mínima de cada noche completa, que es lo que compara la app con 0 °C.

In [ ]:
nombre_base = 'Starter' if ENTRENAR_STARTER else 'GFS + corrección media'
base = pred_starter['V1'] if ENTRENAR_STARTER else pred_corr['V1']
m_va = CORTES['V1'][1]
en = error_nocturno_T(base, Ytr[m_va], fechas[m_va])
print(f'Modelo: {nombre_base} · noches completas en V1: {en["noches_completas"]}')
print(f'Error de la mínima nocturna: {en["minima_todas"]:.2f} °C en todas las celdas · {en["minima_puna"]:.2f} °C en la puna')
print(f'Error horario nocturno:      {en["horario_todas"]:.2f} °C en todas las celdas · {en["horario_puna"]:.2f} °C en la puna')
print(f'Margen ámbar = máx(2 °C, error de la mínima en la puna) = {max(2.0, en["minima_puna"]):.2f} °C')

## 8. Tabla para la bitácora

Copia las filas nuevas a `train/BITACORA.md`. A Kaggle solo se envía un experimento que mejore en V1 y en V2 frente a la mejor línea base.

In [ ]:
def a_markdown(df):
    cols = list(df.columns)
    lineas = ['| ' + ' | '.join(cols) + ' |', '|' + ' --- |' * len(cols)]
    lineas += ['| ' + ' | '.join(str(v) for v in fila) + ' |' for fila in df.astype(str).values]
    return '\n'.join(lineas)

tabla = pd.DataFrame(RESULTADOS)
tabla.to_csv('resultados_validacion.csv', index=False)
print(a_markdown(tabla))

## 9. Cómo medir un modelo nuevo

1. Haz una copia de este notebook.
2. Para cada corte, entrena el modelo **solo** con las filas `CORTES[corte][0]` y calcula sus estadísticas (normalización, climatología, etc.) con esas mismas filas.
3. Predice las filas `CORTES[corte][1]` en las unidades originales: RH en %, T en K, U y V en m/s.
4. Llama a `evaluar('nombre', {'V1': pred_v1, 'V2': pred_v2}, 'qué cambiaste')` y copia la tabla de la sección 8.

Para el enfoque nuevo, el objetivo de la red es la corrección `Ytr - Gtr` y la predicción final es `Gtr[m_va] + corrección predicha`. La línea base que debe superar es «GFS + corrección media» y, sobre todo, el starter.